# YOLO 학습 + INT8 TFLite Export

Google Drive의 `prepared_dataset`으로 YOLO를 학습하고, 라즈베리파이용 `best_int8.tflite`까지 생성합니다.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
!pip install -q ultralytics onnx onnxsim tensorflow


In [ ]:
from pathlib import Path
import shutil
import yaml
import torch
from ultralytics import YOLO

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/yolo_safe_danger')
DRIVE_DATASET_DIR = DRIVE_PROJECT_DIR / 'prepared_dataset'
LOCAL_DATASET_DIR = Path('/content/yolo_safe_danger_prepared_dataset')
RUNS_DIR = DRIVE_PROJECT_DIR / 'runs'

MODEL_SOURCE = 'yolov8n.pt'
RUN_NAME = 'safe_danger_yolo'
IMAGE_SIZE = 416
EPOCHS = 30
BATCH_SIZE = 16
WORKERS = 2
CLASS_NAMES = ['head', 'helmet', 'vest']

print('CUDA:', torch.cuda.is_available())
print('Dataset:', DRIVE_DATASET_DIR)


In [ ]:
if not DRIVE_DATASET_DIR.exists():
    raise FileNotFoundError(f'Dataset folder not found: {DRIVE_DATASET_DIR}')

if LOCAL_DATASET_DIR.exists():
    shutil.rmtree(LOCAL_DATASET_DIR)

shutil.copytree(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)

valid_name = 'valid' if (LOCAL_DATASET_DIR / 'valid' / 'images').exists() else 'val'
if not (LOCAL_DATASET_DIR / valid_name / 'images').exists():
    raise FileNotFoundError('valid/images 또는 val/images 폴더가 없습니다.')

data = {
    'path': str(LOCAL_DATASET_DIR),
    'train': 'train/images',
    'val': f'{valid_name}/images',
    'test': 'test/images' if (LOCAL_DATASET_DIR / 'test' / 'images').exists() else f'{valid_name}/images',
    'nc': len(CLASS_NAMES),
    'names': CLASS_NAMES,
}

LOCAL_YAML = LOCAL_DATASET_DIR / 'data.yaml'
LOCAL_YAML.write_text(yaml.safe_dump(data, sort_keys=False), encoding='utf-8')

print(LOCAL_YAML.read_text())
print('train images:', len(list((LOCAL_DATASET_DIR / 'train' / 'images').glob('*'))))
print('val images:', len(list((LOCAL_DATASET_DIR / valid_name / 'images').glob('*'))))


In [ ]:
model = YOLO(MODEL_SOURCE)

model.train(
    data=str(LOCAL_YAML),
    imgsz=IMAGE_SIZE,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    workers=WORKERS,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    patience=20,
    device=0 if torch.cuda.is_available() else 'cpu',
    plots=True,
)


In [ ]:
best_pt = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'
if not best_pt.exists():
    raise FileNotFoundError(f'best.pt not found: {best_pt}')

model = YOLO(str(best_pt))

model.export(
    format='tflite',
    imgsz=IMAGE_SIZE,
    int8=True,
    data=str(LOCAL_YAML),
)

tflite_files = list((RUNS_DIR / RUN_NAME).rglob('*.tflite'))
if not tflite_files:
    tflite_files = list(Path('/content').rglob('*.tflite'))

print('TFLite files:')
for file in tflite_files:
    print(file)

int8_files = [p for p in tflite_files if 'int8' in p.name.lower()]
if int8_files:
    dst = DRIVE_PROJECT_DIR / 'best_int8.tflite'
    shutil.copy2(int8_files[0], dst)
    print('Copied:', dst)
